# Гайд: AdaBoost и Градиентный бустинг

---

## AdaBoost (Adaptive Boosting)

### Идея
AdaBoost строит сильный классификатор как композицию слабых моделей (обычно пней — деревьев глубины 1).  
Каждый новый классификатор исправляет ошибки предыдущих, увеличивая вес сложных объектов.

---

### Алгоритм по шагам

1. **Инициализация весов**

   $$
   w_i = \frac{1}{l}, \quad i = 1, \dots, l
   $$

   Все объекты изначально одинаково важны.

2. **Для каждой итерации $t = 1, \dots, T$:**

   a) Обучаем базовый классификатор $b_t(x)$ на выборке с весами $w_i$.  
   Он минимизирует взвешенную ошибку:

   $$
   N_t = \sum_i w_i \mathbb{1}\{b_t(x_i) \neq y_i\}
   $$

   b) Вычисляем вес классификатора:

   $$
   \alpha_t = \frac{1}{2} \ln \frac{1 - N_t}{N_t}
   $$

   Чем меньше ошибка, тем больше доверие к модели.

   c) Обновляем веса объектов:

   $$
   w_i \leftarrow w_i \, e^{-\alpha_t y_i b_t(x_i)}
   $$

   Ошибочные примеры получают больший вес.

   d) Нормализуем веса, чтобы сумма $\sum_i w_i = 1$.

3. **Финальный ансамбль**

   $$
   F(x) = \text{sign}\!\left(\sum_{t=1}^{T} \alpha_t b_t(x)\right)
   $$

---

### Пример

Пусть 3 объекта:

| i | xᵢ | yᵢ |
|---|----|----|
| 1 | A  | +1 |
| 2 | B  | −1 |
| 3 | C  | +1 |

#### Шаг 1
Первый пень:

| i | b₁(xᵢ) | Правильно? |
|---|---------|-------------|
| 1 | +1 | да |
| 2 | +1 | нет |
| 3 | +1 | да |

Ошибка $N₁ = 0.333$  
$\alpha₁ = 0.3466$  
Новые веса $w = [0.25, 0.5, 0.25]$.

#### Шаг 2
Второй пень ошибается только на 3-м:

$$
N₂ = 0.25 \Rightarrow \alpha₂ = 0.5493
$$

Новые веса $w = [0.17, 0.33, 0.50]$.

#### Финальная модель

$$
F_2(x) = 0.3466 b_1(x) + 0.5493 b_2(x)
$$

$$
\hat{y}(x) = \text{sign}(F_2(x))
$$

| i | F₂(xᵢ) | sign(F₂(xᵢ)) | Результат |
|---|---------|---------------|------------|
| 1 | +0.90 | +1 | верно |
| 2 | −0.20 | −1 | верно |
| 3 | −0.20 | −1 | ошибка |

---

### Интуиция
- Веса $w_i$ обозначают внимание к трудным примерам.  
- $\alpha_t$ — доверие к каждому классификатору.  
- Итог — взвешенное голосование слабых моделей.

---

## Градиентный бустинг (Gradient Boosting)

### Идея
Gradient Boosting — обобщение AdaBoost.  
Он минимизирует любую дифференцируемую функцию потерь, используя принцип градиентного спуска в пространстве функций.

---

### Алгоритм по шагам

1. **Инициализация модели**

   В начале у нас нет предсказаний, поэтому строим **начальную модель** \(F_0(x)\).  
   Это просто константа, которая минимизирует функцию потерь на всём обучающем наборе:

   $$
   F_0(x) = \arg\min_c \sum_i L(y_i, c)
   $$

   То есть мы ищем одно число \(c\), которое делает среднюю ошибку минимальной.  
   В зависимости от функции потерь это значение имеет конкретный смысл:

   - Для **MSE (регрессия)**:  
     $$
     F_0(x) = \text{mean}(y_i)
     $$

   - Для **логистической потери (классификация)**:  
     $$
     F_0(x) = \frac{1}{2}\ln\frac{p}{1-p}, \quad \text{где } p = P(y=1)
     $$

   - Для **MAE (средняя абсолютная ошибка)**:  
     $$
     F_0(x) = \text{median}(y_i)
     $$

   Таким образом, \(F_0(x)\) — это **первая, грубая аппроксимация** целевой функции.  
   Она одинаковая для всех \(x\) (константа), и дальше на каждой итерации бустинг постепенно улучшает её.


2. **Для каждой итерации $t = 1, \dots, T$:**

   a) Вычисляем остатки (антиградиенты):

   $$
   r_{it} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \Big|_{F = F_{t-1}}
   $$

   Это направление, в котором нужно улучшить предсказание.

   b) Обучаем базовую модель $b_t(x)$ предсказывать эти остатки:

   $$
   b_t(x) \approx r_{it}
   $$

   c) Находим оптимальный шаг $\alpha_t$:

   $$
   \alpha_t = \arg\min_\alpha \sum_i L(y_i, F_{t-1}(x_i) + \alpha b_t(x_i))
   $$

   На практике часто фиксируют learning rate.

   d) Обновляем модель:

   $$
   F_t(x) = F_{t-1}(x) + \alpha_t b_t(x)
   $$

3. **Финальный ансамбль**

   $$
   F_T(x) = \sum_{t=0}^{T} \alpha_t b_t(x)
   $$

---

### Пример (MSE, регрессия)

Функция потерь:

$$
L(y, F(x)) = \frac{1}{2}(y - F(x))^2
$$

Градиент:

$$
\frac{\partial L}{\partial F(x)} = F(x) - y
$$

Остаток (антиградиент):

$$
r_i = y_i - F_{t-1}(x_i)
$$

#### Шаги

1. **Инициализация**

   $$
   F_0(x) = \text{среднее}(y)
   $$

2. **Итерации**
   - Вычисляем остатки $r_i = y_i - F_{t-1}(x_i)$
   - Обучаем дерево $b_t(x) \approx r_i$
   - Обновляем $F_t(x) = F_{t-1}(x) + \alpha b_t(x)$

#### Пример

| i | yᵢ | F₀(xᵢ)=2.0 | Остаток rᵢ | b₁(xᵢ)≈rᵢ | F₁(xᵢ)=F₀+αb₁ |
|---|---|-------------|-------------|-------------|----------------|
| 1 | 4 | 2.0 | 2.0 | 2.0 | 2.2 |
| 2 | 3 | 2.0 | 1.0 | 1.0 | 2.1 |
| 3 | 1 | 2.0 | −1.0 | −1.0 | 1.9 |

Модель шаг за шагом приближает предсказания к истинным значениям.

---

### Интуиция
- Функция потерь говорит, насколько модель ошибается.  
- Остатки — это направление, куда нужно подвинуть предсказания.  
- Каждая новая модель приближает градиент функции потерь.  
- Итоговая модель — сумма поправок от всех деревьев.

---

## Главное различие

| Характеристика | AdaBoost | Gradient Boosting |
|-----------------|-----------|------------------|
| Функция потерь | Экспоненциальная $e^{-yF(x)}$ | Любая дифференцируемая |
| Что предсказывает базовая модель | Метку класса ±1 | Остаток (градиент ошибки) |
| α (альфа) | Вес классификатора | Шаг обучения (learning rate) |
| Тип задачи | Классификация | Классификация / Регрессия |
| Интуиция | Усиливает внимание на трудных примерах | Двигается вдоль градиента потерь |
